In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = '../../data/mixed_population_dataset_160_households.csv'
print(f"Reading data from {DATA_PATH}")

#Converting a CSv into a df
df = pd.read_csv(DATA_PATH)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15,7)

Reading data from ../../data/mixed_population_dataset_160_households.csv


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1382400 entries, 0 to 1382399
Data columns (total 4 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   timestamp      1382400 non-null  object 
 1   consumption_l  1382400 non-null  float64
 2   is_leak        1382400 non-null  int64  
 3   household_id   1382400 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 42.2+ MB


In [3]:
df.describe()

,consumption_l,is_leak
count,1.382400e+06,1.382400e+06
mean,4.485368e+00,2.083333e-03
std,1.209474e+01,4.559599e-02
min,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00
50%,2.121889e-01,0.000000e+00
75%,4.068171e+00,0.000000e+00
max,3.290835e+02,1.000000e+00


In [3]:
#Ensure the timestamp from the excel is datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['day_of_month'] = df['timestamp'].dt.day
df['week_of_year'] = df['timestamp'].dt.isocalendar().week.astype(int)
df['month'] = df['timestamp'].dt.month

df['is_weekend'] = (df['day_of_week'] >=5).astype(int)

In [4]:
print(df.head(5))

            timestamp  consumption_l  is_leak  \
0 2025-11-03 00:00:00       2.185477        0   
1 2025-11-03 00:15:00       0.000000        0   
2 2025-11-03 00:30:00       0.000000        0   
3 2025-11-03 00:45:00       0.000000        0   
4 2025-11-03 01:00:00       0.000000        0   

                      household_id  hour  day_of_week  day_of_month  \
0  household_apartment_family_4855     0            0             3   
1  household_apartment_family_4855     0            0             3   
2  household_apartment_family_4855     0            0             3   
3  household_apartment_family_4855     0            0             3   
4  household_apartment_family_4855     1            0             3   

   week_of_year  month  is_weekend  
0            45     11           0  
1            45     11           0  
2            45     11           0  
3            45     11           0  
4            45     11           0  


En el siguiente approach en vez de usar el train_test_split, que baraja los datos de forma aleatoria antes de dividirlos. En mi caso tengo una serie temporal conologica. Al barajar mezclo el pasado y el futuro.

Para una mejor evaluacion voy a dividir los datos cronologicamente de forma manual.

In [5]:

print("Preparing the data for training")

features = [
    'consumption_l',
    'hour',
    'day_of_week',
    'week_of_year',
    'month',
    'is_weekend'
]

target = 'is_leak'


df = df.sort_values('timestamp')
#split, 80% training 20%testing
split_point = int(len(df) * 0.8)

train_df = df.iloc[:split_point]
test_df = df.iloc[split_point:]

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

print("Split done")


Preparing the data for training
Split done


Aplicar un escalado a los datos despues de la division para evitar fuga de datos

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
from sklearn.ensemble import IsolationForest

print("Training the Model")

contamination_rate = y_train.mean()

print(f"Estimated pollution rate: {contamination_rate:.4f}")

iso_forest_tuned = IsolationForest(
    n_estimators=100,
    contamination = contamination_rate,
    random_state=42,
    n_jobs=-1
)

iso_forest_tuned.fit(X_train_scaled)
print("Training done")

Training the Model
Estimated pollution rate: 0.0020
Training done


In [14]:
# --- CICLO DE AJUSTE ---

# 1. Elige un nuevo valor, más bajo.
new_contamination = 0.001 # O 0.0008, o 0.0015... ¡Experimenta!

# 2. Re-entrena el modelo con este nuevo valor
iso_forest_calibrated = IsolationForest(
    n_estimators=100,
    contamination=new_contamination, # <-- El nuevo valor
    random_state=42
)
iso_forest_calibrated.fit(X_train)

# 3. Predice y evalúa de nuevo
y_pred_calibrated = iso_forest_calibrated.predict(X_test)
y_pred_mapped_calibrated = np.where(y_pred_calibrated == -1, 1, 0)

print(f"\nResultados con contamination = {new_contamination}")
print(f"Número de fugas predichas: {np.sum(y_pred_mapped_calibrated)}")
print(f"Número real de fugas en el conjunto de prueba: {y_test.sum()}")

# 4. Imprime el reporte de clasificación para ver el efecto en precision y recall
print(classification_report(y_test, y_pred_mapped_calibrated, target_names=['No Fuga (0)', 'Fuga (1)']))


Resultados con contamination = 0.001
Número de fugas predichas: 1755
Número real de fugas en el conjunto de prueba: 640


NameError: name 'classification_report' is not defined

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt

# Crearemos un DataFrame temporal con los datos de prueba para facilitar la visualización
plot_df = X_test.copy()
plot_df['is_leak'] = y_test

plt.figure(figsize=(12, 6))
sns.histplot(data=plot_df, x='consumption_l', hue='is_leak', multiple='stack', bins=100)
plt.title('Distribución del Consumo para Fugas vs. No Fugas')
plt.xlabel('Consumo (L)')
plt.yscale('log') # Usamos escala logarítmica para ver mejor las barras pequeñas
plt.show()

# También un boxplot puede ser muy revelador
plt.figure(figsize=(12, 6))
sns.boxplot(data=plot_df, x='is_leak', y='consumption_l')
plt.title('Comparación de Consumo para Fugas vs. No Fugas')
plt.xlabel('¿Es Fuga?')
plt.ylabel('Consumo (L)')
plt.ylim(0, plot_df['consumption_l'].quantile(0.99)) # Cortamos valores extremos para ver mejor el cuerpo de la distribución
plt.show()

NameError: name 'X_test' is not defined

In [18]:
# --- CICLO DE AJUSTE ---
from sklearn.metrics import classification_report, confusion_matrix
# 1. Elige un nuevo valor, más bajo.
new_contamination = 0.0008 # O 0.0008, o 0.0015... ¡Experimenta!

# 2. Re-entrena el modelo con este nuevo valor
iso_forest_calibrated_1 = IsolationForest(
    n_estimators=100,
    contamination=new_contamination, # <-- El nuevo valor
    random_state=42
)
iso_forest_calibrated_1.fit(X_train)

# 3. Predice y evalúa de nuevo
y_pred_calibrated = iso_forest_calibrated_1.predict(X_test)
y_pred_mapped_calibrated = np.where(y_pred_calibrated == -1, 1, 0)

print(f"\nResultados con contamination = {new_contamination}")
print(f"Número de fugas predichas: {np.sum(y_pred_mapped_calibrated)}")
print(f"Número real de fugas en el conjunto de prueba: {y_test.sum()}")

# 4. Imprime el reporte de clasificación para ver el efecto en precision y recall
print(classification_report(y_test, y_pred_mapped_calibrated, target_names=['No Fuga (0)', 'Fuga (1)']))


Resultados con contamination = 0.0008
Número de fugas predichas: 1393
Número real de fugas en el conjunto de prueba: 640
              precision    recall  f1-score   support

 No Fuga (0)       1.00      0.99      1.00    275840
    Fuga (1)       0.00      0.00      0.00       640

    accuracy                           0.99    276480
   macro avg       0.50      0.50      0.50    276480
weighted avg       1.00      0.99      0.99    276480



In [19]:
import pandas as pd

# Obtener los scores de anomalía para TODO el conjunto de prueba
anomaly_scores = iso_forest_calibrated_1.decision_function(X_test)

# Crear un DataFrame para analizar los resultados
results_df = pd.DataFrame(X_test).copy()
results_df['anomaly_score'] = anomaly_scores
results_df['is_leak_real'] = y_test
results_df['is_leak_predicted'] = y_pred_mapped_calibrated

# Analicemos los scores de las fugas REALES
real_leaks_scores = results_df[results_df['is_leak_real'] == 1]['anomaly_score']

# Analicemos los scores de las predicciones FALSAS del modelo
false_alarms_scores = results_df[results_df['is_leak_predicted'] == 1]['anomaly_score']

print("\n--- Análisis de Scores de Anomalía ---")
print("Estadísticas de scores para FUGAS REALES:")
print(real_leaks_scores.describe())

print("\nEstadísticas de scores para FALSAS ALARMAS (Predicciones del Modelo):")
print(false_alarms_scores.describe())


--- Análisis de Scores de Anomalía ---
Estadísticas de scores para FUGAS REALES:
count    640.000000
mean       0.114466
std        0.028984
min       -0.014737
25%        0.098066
50%        0.120297
75%        0.135803
max        0.156952
Name: anomaly_score, dtype: float64

Estadísticas de scores para FALSAS ALARMAS (Predicciones del Modelo):
count    1393.000000
mean       -0.009918
std         0.007805
min        -0.035585
25%        -0.015839
50%        -0.007961
75%        -0.003525
max        -0.000021
Name: anomaly_score, dtype: float64


In [20]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report

# Solo usamos la columna de consumo
features_simple = ['consumption_l']
X_train_simple = X_train[features_simple]
X_test_simple = X_test[features_simple]

# Usamos la contaminación original como punto de partida
contamination_rate = y_train.mean()

iso_forest_simple = IsolationForest(
    contamination=contamination_rate,
    random_state=42
)

print("\n--- Entrenando modelo simple (solo consumo) ---")
iso_forest_simple.fit(X_train_simple)

# Predecir y evaluar
y_pred_simple = iso_forest_simple.predict(X_test_simple)
y_pred_mapped_simple = np.where(y_pred_simple == -1, 1, 0)

print(f"Número de fugas predichas: {np.sum(y_pred_mapped_simple)}")
print(f"Número real de fugas: {y_test.sum()}")
print(classification_report(y_test, y_pred_mapped_simple))


--- Entrenando modelo simple (solo consumo) ---
Número de fugas predichas: 557
Número real de fugas: 640
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    275840
           1       0.01      0.00      0.01       640

    accuracy                           1.00    276480
   macro avg       0.50      0.50      0.50    276480
weighted avg       1.00      1.00      1.00    276480



In [17]:
# --- CICLO DE AJUSTE ---

# 1. Elige un nuevo valor, más bajo.
new_contamination = 0.0015 # O 0.0008, o 0.0015... ¡Experimenta!

# 2. Re-entrena el modelo con este nuevo valor
iso_forest_calibrated_2 = IsolationForest(
    n_estimators=100,
    contamination=new_contamination, # <-- El nuevo valor
    random_state=42
)
iso_forest_calibrated_2.fit(X_train)

# 3. Predice y evalúa de nuevo
y_pred_calibrated = iso_forest_calibrated_2.predict(X_test)
y_pred_mapped_calibrated = np.where(y_pred_calibrated == -1, 1, 0)

print(f"\nResultados con contamination = {new_contamination}")
print(f"Número de fugas predichas: {np.sum(y_pred_mapped_calibrated)}")
print(f"Número real de fugas en el conjunto de prueba: {y_test.sum()}")

# 4. Imprime el reporte de clasificación para ver el efecto en precision y recall
print(classification_report(y_test, y_pred_mapped_calibrated, target_names=['No Fuga (0)', 'Fuga (1)']))


Resultados con contamination = 0.0015
Número de fugas predichas: 2661
Número real de fugas en el conjunto de prueba: 640


NameError: name 'classification_report' is not defined

In [13]:
# Realizar predicciones en el conjunto de prueba
y_pred_test = iso_forest_tuned.predict(X_test)

# Los valores predichos son -1 y 1. Necesitamos mapearlos a 1 y 0 para compararlos con y_test.
# Anomalía (-1) -> Fuga (1)
# Normal (1) -> No Fuga (0)

y_pred_mapped = np.where(y_pred_test == -1, 1, 0)

print("\nPredicciones realizadas en el conjunto de prueba.")
print(f"Número de fugas predichas: {np.sum(y_pred_mapped)}")
print(f"Número real de fugas en el conjunto de prueba: {y_test.sum()}")


Predicciones realizadas en el conjunto de prueba.
Número de fugas predichas: 3661
Número real de fugas en el conjunto de prueba: 640
